In [ ]:
pip install torch transformers tensorflow camel-tools


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
# Charger le modèle AraGPT2
MODEL_NAME = "aubmindlab/aragpt2-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("Modèle et tokenizer chargés avec succès !")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/553M [00:00<?, ?B/s]

Modèle et tokenizer chargés avec succès !


In [ ]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import normalize_alef_maksura_ar, normalize_teh_marbuta_ar, normalize_alef_ar


In [ ]:
def preprocess_arabic_text(text):
    """Normalise le texte arabe pour une meilleure cohérence."""
    text = dediac_ar(text)  # Supprime les diacritiques
    text = normalize_alef_ar(text)  # Normalise les variantes d'alef
    text = normalize_alef_maksura_ar(text)  # Normalise 'ى' → 'ي'
    text = normalize_teh_marbuta_ar(text)  # Normalise 'ة' → 'ه'
    return text

In [ ]:
# Exemple
input_text = "مَرْحَباً كيف حالك؟"
clean_text = preprocess_arabic_text(input_text)
print("Texte normalisé :", clean_text)

Texte normalisé : مرحبا كيف حالك؟


In [ ]:
def generate_response(input_text, max_length=50, temperature=0.7):
   # Prétraiter l'entrée utilisateur
    input_text = preprocess_arabic_text(input_text)

    # Ajouter un token de début si nécessaire
    input_ids = tokenizer.encode(input_text, return_tensors="pt")

    # Génération
    output_ids = model.generate(
        input_ids,
        max_length=max_length,
        temperature=temperature,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=2
    )

    # Décodage de la réponse
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return response

In [ ]:
# Test avec une question
user_input = "كيف يمكنني تعلم البرمجة؟"
response = generate_response(user_input)
print("Chatbot :", response)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Chatbot : كيف يمكنني تعلم البرمجه؟ ؟ ؟!! ؟.. ؟ هل تعلم أن هناك الكثير من البرامج التي تعمل على تطوير المواقع ، ولكن لا يوجد برنامج واحد على الإطلاق ، بل هناك العديد من المواقع التي تقدم خدمات تطوير مواقع ، مثل : [


In [ ]:
# Fonction pour l'interface Gradio
def chatbot_interface(user_input):
    return generate_response(user_input)


In [ ]:
# Configuration de l'interface avec Gradio
interface = gr.Interface(
    fn=chatbot_interface,
    inputs=gr.Textbox(lines=5, placeholder="اكتب سؤالك هنا..."),
    outputs="text",
    title="روبوت دردشة باللغة العربية",
    description="مرحباً! أنا روبوت دردشة. اكتب سؤالك باللغة العربية وسأحاول الإجابة."
)

In [ ]:
# Lancer l'interface web
interface.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bd8fe7203a2868fafa.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import files
uploaded = files.upload()

In [25]:
!python chatbot_arabe_gradio.ipynb

python3: can't open file '/content/chatbot_arabe_gradio.ipynb': [Errno 2] No such file or directory


In [26]:
!ls


sample_data


In [ ]:
print("مرحباً! أنا روبوت دردشة. كيف يمكنني مساعدتك اليوم؟")
print("اكتب 'خروج' لإنهاء المحادثة.\n")

while True:
    # Entrée utilisateur
    user_input = input("أنت: ")
    if user_input.strip().lower() in ["خروج", "انهاء"]:
        print("روبوت: وداعاً! سعدت بالتحدث معك.")
        break

    # Réponse du chatbot
    response = generate_response(user_input)
    print("روبوت:", response)


مرحباً! أنا روبوت دردشة. كيف يمكنني مساعدتك اليوم؟
اكتب 'خروج' لإنهاء المحادثة.

أنت: خروج
روبوت: وداعاً! سعدت بالتحدث معك.


In [ ]:
pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.9/321.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2
